# Notebook 12.1  Cascade vs. unified Arabic-to-English translation, and a small spoken-language-understanding parser

**Goal.** Compare a cascade and a unified model on a small Arabic-to-English set with BLEU and a neural-style metric, build a simple intent-and-slot parser for spoken Arabic commands, and simulate a wait-k policy to compute Average Lagging.

**What runs here.** The metrics (BLEU, a cosine sentence-similarity proxy for COMET, intent accuracy, slot F1) and the wait-k / Average Lagging simulation are real and run with no downloads. The translation and recognition steps are hooks with synthetic transcripts/translations so the notebook always runs; each step says how to plug in Whisper, a machine-translation model, or SeamlessM4T. This accompanies Chapter 12.

## 1. Setup

In [ ]:
import re, math
from collections import Counter
print('ready')

## 2. A tiny Arabic-to-English evaluation set

Each item has the source Arabic, a reference English translation, and two system outputs: one from a *cascade* (recognizer then translator) and one from a *unified* model. With real systems you would generate the `cascade` and `unified` fields by calling them; here they are provided so the metrics run.

In [ ]:
data = [
  {'src':'أين أقرب محطة وقود',      'ref':'where is the nearest gas station',
   'cascade':'where is the nearest gas station',     'unified':'where is the closest fuel station'},
  {'src':'أريد حجز رحلة إلى الرياض', 'ref':'i want to book a flight to riyadh',
   'cascade':'i want to book a trip to riyadh',      'unified':'i want to book a flight to riyadh'},
  {'src':'كم سعر هذا',               'ref':'how much is this',
   'cascade':'how much this',                        'unified':'how much is this'},
  {'src':'لا أوافق على الشروط',      'ref':'i do not agree to the terms',
   'cascade':'i agree to the terms',                 'unified':'i do not agree to the terms'},
]
print('items:', len(data))

## 3. BLEU and a sentence-similarity proxy

A compact BLEU (up to 4-grams, with a brevity penalty) and a cosine bag-of-words similarity that stands in for a neural metric such as COMET. The cascade item that drops the negation (“i agree” for “i do not agree”) shows why lexical overlap alone can miss a meaning error.

In [ ]:
def ngrams(toks,n): return Counter(tuple(toks[i:i+n]) for i in range(len(toks)-n+1))
def bleu(ref,hyp,N=4):
    r,h=ref.split(),hyp.split()
    if not h: return 0.0
    weights=[]
    for n in range(1,N+1):
        hn,rn=ngrams(h,n),ngrams(r,n)
        if not hn: weights.append(0.0); continue
        overlap=sum(min(c,rn.get(g,0)) for g,c in hn.items())
        weights.append(overlap/max(1,sum(hn.values())))
    if min(weights)==0: gm=0.0
    else: gm=math.exp(sum(math.log(w) for w in weights)/N)
    bp=1.0 if len(h)>len(r) else math.exp(1-len(r)/max(1,len(h)))  # brevity penalty
    return bp*gm

def cos_sim(a,b):
    A,B=Counter(a.split()),Counter(b.split())
    common=set(A)&set(B)
    dot=sum(A[w]*B[w] for w in common)
    na=math.sqrt(sum(v*v for v in A.values())); nb=math.sqrt(sum(v*v for v in B.values()))
    return dot/(na*nb) if na and nb else 0.0

for sysname in ['cascade','unified']:
    b=sum(bleu(d['ref'],d[sysname]) for d in data)/len(data)
    s=sum(cos_sim(d['ref'],d[sysname]) for d in data)/len(data)
    print(f'{sysname:8s}  BLEU={b:.3f}  sim={s:.3f}')
print('\nnote: the cascade drops a negation in item 4 ("i agree" vs "i do not agree"),')
print('a meaning error that lexical overlap penalizes only lightly, so report human checks too.')

## 4. A spoken-command intent-and-slot parser

A tiny rule-based parser maps Arabic commands to an intent and slots, the cascaded-SLU idea (recognize, then interpret the text). With a real system you would replace the rules with a trained classifier or an LLM prompt that returns JSON. We then score intent accuracy and slot F1 against the ground truth.

In [ ]:
commands = [
  {'text':'احجز رحلة إلى الرياض غدا',  'intent':'book_flight','slots':{'destination':'الرياض','date':'غدا'}},
  {'text':'احجز رحلة إلى جدة اليوم',   'intent':'book_flight','slots':{'destination':'جدة','date':'اليوم'}},
  {'text':'ما حالة الرحلة',            'intent':'flight_status','slots':{}},
  {'text':'الغ الحجز',                'intent':'cancel_booking','slots':{}},
]
CITIES={'الرياض','جدة','الدمام','مكة','المدينة'}; DATES={'غدا','اليوم','بكرا'}
def parse(text):
    toks=text.split()
    if 'احجز' in toks: intent='book_flight'
    elif 'حالة' in toks: intent='flight_status'
    elif 'الغ' in toks or 'إلغاء' in toks: intent='cancel_booking'
    else: intent='unknown'
    slots={}
    for t in toks:
        if t in CITIES: slots['destination']=t
        if t in DATES: slots['date']=t
    return intent,slots

def slot_f1(gold,pred):
    g=set(gold.items()); p=set(pred.items())
    tp=len(g&p); fp=len(p-g); fn=len(g-p)
    prec=tp/(tp+fp) if tp+fp else 1.0; rec=tp/(tp+fn) if tp+fn else 1.0
    return 2*prec*rec/(prec+rec) if prec+rec else 0.0

correct=0; f1s=[]
for c in commands:
    pi,ps=parse(c['text'])
    correct+= (pi==c['intent'])
    f1s.append(slot_f1(c['slots'],ps))
    print(f"{c['text']}  ->  intent={pi}  slots={ps}")
print(f'\nintent accuracy = {correct/len(commands):.2f}   mean slot F1 = {sum(f1s)/len(f1s):.2f}')

## 5. Simultaneous translation: wait-k and Average Lagging

Simulate a wait-k policy (k=3): the source arrives one word per second, and the system emits one target word per step starting after k source words. Average Lagging is the mean amount by which the output trails an ideal speaker-synchronized output.

In [ ]:
def average_lagging(source_len, k):
    # emit one target token per step after waiting k source tokens; target length ~ source length
    tgt_len = source_len
    # g(t): number of source tokens read when emitting target token t (1-indexed), capped at source_len
    lags=[]
    r = tgt_len/source_len  # = 1 here
    for t in range(1, tgt_len+1):
        g = min(k + (t-1), source_len)      # source words read when emitting token t
        lags.append(g - (t-1)/r)            # AL per-step term
    # average over tokens until source is exhausted (standard AL truncation)
    tau = next((t for t in range(1,tgt_len+1) if min(k+(t-1),source_len)==source_len), tgt_len)
    return sum(lags[:tau])/tau

for k in [1,3,5]:
    print(f'k={k}:  Average Lagging = {average_lagging(8,k):.2f} words')
print('\nlarger k waits longer (higher Average Lagging) but usually translates better; report AL with quality.')

## 6. Where to go next

- Replace the provided `cascade`/`unified` outputs with real systems: Whisper plus a machine-translation model for the cascade, and SeamlessM4T for the unified model; score with sacreBLEU and COMET.
- Swap the rule-based parser for a trained classifier or an LLM prompt that returns JSON intents and slots, and report invalid-JSON, missing-slot, and wrong-slot-value rates.
- Evaluate on dialectal and code-switched test sets (for example TARIC-SLU and TEDxTN), and report metrics per dialect.
- For streaming, measure Average Lagging in time over real audio chunks, not only in word steps.